In [1]:
import pandas as pd
import numpy as np
import boto3

In [2]:
df = pd.read_csv("AB_NYC_2019.csv")
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [3]:
print(df.shape)
print(df.columns.tolist())
df.info()
df.isnull().sum()

(48895, 16)
['id', 'name', 'host_id', 'host_name', 'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  object 
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  object 
 4   neighbourhood_group             48895 non-null  object 
 5   neighbourhood                   48895 non-null  object 
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type

id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64

In [4]:
# Remove duplicates
df = df.drop_duplicates()

# Drop columns not useful for prediction
df = df.drop(columns=["id", "name", "host_id", "host_name", "last_review"], errors="ignore")

# Fill missing values
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

# Drop rows missing key fields
df = df.dropna(subset=["neighbourhood_group", "room_type", "price"])

# Remove extreme outliers in price
df = df[(df["price"] > 0) & (df["price"] <= 500)]

print(df.shape)
df.isnull().sum()

(47840, 11)


neighbourhood_group               0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    0
number_of_reviews                 0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
dtype: int64

In [5]:
import numpy as np

# Create price category (for analysis / write-up)
df["price_category"] = pd.cut(
    df["price"],
    bins=[0, 100, 200, 500],
    labels=["low", "medium", "high"]
)

# Log transform price (helps model stability)
df["log_price"] = np.log1p(df["price"])

# Convert categorical variables to numbers
df = pd.get_dummies(df, columns=["neighbourhood_group", "room_type"], drop_first=True)

df.head()

,neighbourhood,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,price_category,log_price,neighbourhood_group_Brooklyn,neighbourhood_group_Manhattan,neighbourhood_group_Queens,neighbourhood_group_Staten Island,room_type_Private room,room_type_Shared room
0,Kensington,40.64749,-73.97237,149,1,9,0.21,6,365,medium,5.010635,True,False,False,False,True,False
1,Midtown,40.75362,-73.98377,225,1,45,0.38,2,355,high,5.420535,False,True,False,False,False,False
2,Harlem,40.80902,-73.94190,150,3,0,0.00,1,365,medium,5.017280,False,True,False,False,True,False
3,Clinton Hill,40.68514,-73.95976,89,1,270,4.64,1,194,low,4.499810,True,False,False,False,False,False
4,East Harlem,40.79851,-73.94399,80,10,9,0.10,1,0,low,4.394449,False,True,False,False,False,False


In [6]:
df.shape

(47840, 17)

In [7]:
X = df.drop(columns=["price", "price_category"], errors="ignore")
y = df["price"]

print(X.shape)
print(y.shape)

(47840, 15)
(47840,)


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 38272
Testing rows: 9568


In [10]:
# Check which columns are still non-numeric
print(X.dtypes[X.dtypes == "object"])

neighbourhood    object
dtype: object


In [12]:
df = pd.get_dummies(
    df,
    columns=["neighbourhood"],
    drop_first=True
)

df.shape

(47840, 234)

In [13]:
X = df.drop(columns=["price", "price_category"], errors="ignore")
y = df["price"]

print(X.shape)
print(y.shape)

(47840, 232)
(47840,)


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 38272
Testing rows: 9568


In [19]:
print(X.columns)

Index(['latitude', 'longitude', 'minimum_nights', 'number_of_reviews',
       'reviews_per_month', 'calculated_host_listings_count',
       'availability_365', 'log_price', 'neighbourhood_group_Brooklyn',
       'neighbourhood_group_Manhattan',
       ...
       'neighbourhood_Westchester Square', 'neighbourhood_Westerleigh',
       'neighbourhood_Whitestone', 'neighbourhood_Williamsbridge',
       'neighbourhood_Williamsburg', 'neighbourhood_Willowbrook',
       'neighbourhood_Windsor Terrace', 'neighbourhood_Woodhaven',
       'neighbourhood_Woodlawn', 'neighbourhood_Woodside'],
      dtype='object', length=232)


In [21]:
df = df.drop(columns=["log_price"], errors="ignore")

In [22]:
X = df.drop(columns=["price", "price_category"], errors="ignore")
y = df["price"]

print([col for col in X.columns if "price" in col])

[]


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [24]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [25]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R2 Score:", r2)

RMSE: 60.66502926169215
R2 Score: 0.5235937922493447
